## 1 - Configurações Gerais

In [1]:
#@markdown Detectando o Ambiente de Execução { display-mode: "form" }
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"AMBIENTE DE EXECUÇÃO DETECTADO: {EXECUTION_ENV}")

AMBIENTE DE EXECUÇÃO DETECTADO: colab


In [2]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q minigrid==3.0.0
    !pip install -q langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.7/136.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 18.8 MB/s eta 0:00:00


In [5]:
import os
import sys

if EXECUTION_ENV == "colab":
    repository_path = os.path.join(os.getcwd(), "minigrid_benchmark")

if EXECUTION_ENV == "kaggle":
    repository_path = os.path.join("/kaggle/working/", "minigrid_benchmark")

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(repository_path):
        !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git
    repository_src_path = os.path.join(repository_path, "src")
    sys.path.append(repository_src_path)
    print(f"Repository src path: {repository_src_path}")

Cloning into 'minigrid_benchmark'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 229 (delta 153), reused 218 (delta 148), pack-reused 0 (from 0)
Receiving objects: 100% (229/229), 436.38 KiB | 7.52 MiB/s, done.
Resolving deltas: 100% (153/153), done.
Repository src path: /content/minigrid_benchmark/src


In [6]:
from langchain_openai import ChatOpenAI

from react_agent import ReActAgent
from wrappers import SYSTEM_PROMPT_WRAPPER_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE, OBS_TEMPLATE_ENG

import experiments_util
from experiments_util import run_and_save_experiments, wrapper1, wrapper2_with_numbers, wrapper2_without_numbers

In [7]:
if EXECUTION_ENV == "colab":
    # Mount Google Drive, to save results there
    from google.colab import drive
    drive.mount('/content/drive')
    results_dir = '/content/drive/My Drive/EAD-Pesquisa-Agentes/_projeto_minigrid/results'

if EXECUTION_ENV == "kaggle":
    results_dir = "/kaggle/working/results"

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    experiments_util.RESULTS_BASE_DIR = results_dir

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
OPENAI_KEY_NAME = "OPENAI_PABLO"
OPENAI_API_KEY = None

if EXECUTION_ENV == "colab":
    # just assert that HUGGINGFACE_API_KEY or KF_TOKEN secret is set
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get(OPENAI_KEY_NAME)

if EXECUTION_ENV == "kaggle":
    # para o Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret(OPENAI_KEY_NAME)

if EXECUTION_ENV == "local":
    OPENAI_API_KEY = os.getenv(OPENAI_KEY_NAME)

if not OPENAI_API_KEY:
    raise ValueError(f"{OPENAI_KEY_NAME} environment variable is not set")

## 2 - Configurar o Modelo

In [9]:
MODEL_GPT_5_4 = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=OPENAI_API_KEY,
)

MODEL_GPT_4_1 = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=OPENAI_API_KEY,
)

## 3 - Configurações do Experimento

In [ ]:

configs = [
    {
        'name': 'Prompt_1_GPT5.4',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT5.4',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
    {
        'name': 'Prompt_1_GPT4.1',
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT4.1',
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
    {
        'name': 'Prompt_1_GPT5.4_with_history',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, history_window=5, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT5.4_with_history',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
        {
        'name': 'Prompt_1_GPT4.1_with_history',
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, history_window=5, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT4.1_with_history',
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
]

## 4 - Execução do Experimento

In [11]:
experiment_name = "2026-05-09_openai"  # se já existir, continua de onde parou

In [12]:
##########################
##  Execute experiments ##
##########################

final_results, filepath = run_and_save_experiments(configs, experiment_name=experiment_name, verbose=True)



In [13]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    # zip the final results directory
    import shutil

    experiment_result_dir = os.path.dirname(filepath)
    zip_path = os.path.join(experiments_util.RESULTS_BASE_DIR, f"{experiment_name}_results_zip")

    # Creates results_zip.zip containing everything
    shutil.make_archive(zip_path, 'zip', experiment_result_dir)

    print(f"Created: {zip_path}.zip")

Created: /content/drive/My Drive/EAD-Pesquisa-Agentes/_projeto_minigrid/results/2026-05-09_openai_results_zip.zip


## Plot Results

In [ ]:
# depois: criar uma maneira fácil de plotar

In [14]:
print("Now, run notebook \"run_experiments_analysis.ipynb\".")

Now, run notebook "run_experiments_analysis.ipynb".
